In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
import pyarrow.compute as pc
from pathlib import Path
import re

# ============================ 参数区 ============================
RAW_FILE = Path(r"D:\毕设数据\LFP\METABatt_A123_APR18650M1B_007.parquet")
# —— 测试回退（在本机找不到时尝试，可删）——
if not RAW_FILE.exists():
    for _alt in [Path("/mnt/user-data/uploads/METABatt_A123_APR18650M1B_007.parquet"),
                 Path("/mnt/user-data/uploads/METABatt_A123_APR18650M1B_007_1.parquet")]:
        if _alt.exists(): RAW_FILE=_alt; break

CHEMISTRY="LFP"
ACN1_A=1.2                      # 1C 电流 = 台架 ACN1（脉冲 2.4/4.8/7.2 = 2/4/6×ACN1）
NOMINAL_CAPACITY_AH=1.2         # SOH 参考容量（额定）
SOH_REFERENCE="rated"           # "rated" -> /1.2 ; "bol" -> /首个check-up容量
SOC_ORDER=["90%","50%","10%"]
NOMINAL_CURRENT_LEVELS_A=np.array([2.4,4.8,7.2])   # 2/4/6 C
PULSE_TARGETS=["PUL","PUL*RES"] # PUL=放电脉冲, PUL*RES=充电脉冲
PULSE_PROCEDURE="jri_APR_CU_Pulse"
BLOCK_GAP_MIN=120.0             # BM 内脉冲间隔 > 此值 = 下一 SOC 块
ZERO_CURRENT_LIMIT=1e-6

# —— R0/R1/R2（移植 NCA，勿改定义）——
R0_TARGET_AFTER_PAUSE_SEC=0.5; R1_ANCHOR_SEC=10.0; R2_ANCHOR_SEC=20.0
R0_FIT_POINT_START=2; R0_FIT_POINT_END=6
FIRST_TWO_VOLTAGE_EQUAL_ATOL=1e-9; VOLTAGE_JUMP_LIMIT=1e-3
MAX_PAUSE_TO_PULSE_GAP_SEC=2.0; CC_CURRENT_DEVIATION_RATIO=0.03
STD_LIMIT={2.4:0.1,4.8:0.1,7.2:0.1}
CC_VALID,CC_INSUF,CC_DEV,CC_NE="Valid","Insufficient duration","Current deviation > 3%","Not evaluated"

# —— load-case 删除 ——
DROP_FIRST_PULSE=True           # ① 每块首条 DCH-2C
APPLY_LOADCASE_REMOVAL=True     # ② Barai 极端工况
HIGH_CRATE_LEVELS=[4,6]         # Barai 意义的"大倍率"

NEEDED_COLS=["Time","Current","Voltage","Temperature","Zustand","target","Prozedur","BM_Programm","ID"]

# v3：删除 CC_10s_Status / CC_20s_Status / R0_Target_Time 三列
OUTPUT_COLUMNS=["SOH","SOC","Battery_ID","Time","ID","Zustand","Zustand/Current",
    "Current","Voltage","Nominal_Current_A","C_rate","chemistry",
    "R0","R1","R2","R0_mOhm","R1_mOhm","R2_mOhm",
    "CC_Valid_10s","CC_Valid_20s",
    "CC_Max_Deviation_10s_pct","CC_Max_Deviation_20s_pct",
    "Pause_to_Pulse_Time_Diff_s","R0_Quality",
    "load_case","load_case_in_scope"]

In [2]:
# ============================ 读取（内存安全）============================
def parse_battery_id(path):
    m=re.search(r"(A123_[A-Za-z0-9]+_\d+)", Path(path).name)
    return m.group(1) if m else Path(path).stem

def build_capacity_map(path):
    pf=pq.ParquetFile(path); cap={}
    for i in range(pf.num_row_groups):
        d=pf.read_row_group(i,columns=["BM_Programm","Capacity_py"]).to_pandas()
        d=d[d["Capacity_py"].notna()]
        for b,g in d.groupby("BM_Programm"): cap.setdefault(int(b),float(g["Capacity_py"].iloc[0]))
    return cap

def read_filtered(path, targets=None, proc=None, cols=None):
    pf=pq.ParquetFile(path); parts=[]
    for i in range(pf.num_row_groups):
        t=pf.read_row_group(i,columns=cols)
        if proc is not None: 
            t=t.filter(pc.equal(t["Prozedur"],proc))
        if targets is not None: 
            t=t.filter(pc.is_in(t["target"],value_set=pa.array(targets)))
        if t.num_rows: 
            parts.append(t.to_pandas())
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

cap_map=build_capacity_map(RAW_FILE)
BOL_CAP=cap_map[min(cap_map)]
SOH_REF_AH=NOMINAL_CAPACITY_AH if SOH_REFERENCE=="rated" else BOL_CAP
BATTERY_ID=parse_battery_id(RAW_FILE)
print("电芯:",BATTERY_ID,"| check-up 数:",len(cap_map),"| SOH 参考容量:",round(SOH_REF_AH,4),"Ah")

电芯: A123_APR18650M1B_007 | check-up 数: 23 | SOH 参考容量: 1.2 Ah


In [3]:
# ============================ 筛选层：脉冲清单（SOC/SOH/load_case + 删除 flag）============================
def build_pulse_inventory(path, cap_map, soh_ref_ah):
    pp=read_filtered(path, targets=PULSE_TARGETS, proc=PULSE_PROCEDURE, cols=NEEDED_COLS)
    # float32 -> float64（关键：否则档位判定会因浮点误差全部失败）
    pp["Time"]=pd.to_datetime(pp["Time"],utc=True)
    pp["Current"]=pd.to_numeric(pp["Current"],errors="coerce").astype(float)
    pp["Voltage"]=pd.to_numeric(pp["Voltage"],errors="coerce").astype(float)
    z=pp["Zustand"].astype(str)
    pp["Zustand"]=np.where(z.str.startswith("DCH"),"DCH",np.where(z.str.startswith("CHA"),"CHA",z))
    pp=pp.dropna(subset=["Time","Current"]).sort_values(["BM_Programm","Time"]).reset_index(drop=True)
    pp["pulse_uid"]=pp["BM_Programm"].astype(str)+"|"+pp["ID"].astype(str)

    g=pp.groupby("pulse_uid",sort=False)
    inv=pd.DataFrame({"BM_Programm":g["BM_Programm"].first(),"ID":g["ID"].first(),
        "Zustand":g["Zustand"].first(),"I_peak":g["Current"].apply(lambda s:s.abs().max()),
        "t0":g["Time"].min()}).reset_index()
    lv=NOMINAL_CURRENT_LEVELS_A
    inv["Nominal_Current_A"]=inv["I_peak"].apply(lambda x:float(lv[np.argmin(np.abs(lv-x))]))
    inv["C_rate"]=(inv["Nominal_Current_A"]/ACN1_A).round().astype(int)
    inv=inv.sort_values(["BM_Programm","t0"]).reset_index(drop=True)
    gap=inv.groupby("BM_Programm")["t0"].diff().dt.total_seconds()/60.0
    newblk=(inv["BM_Programm"]!=inv["BM_Programm"].shift())|(gap>BLOCK_GAP_MIN)
    inv["blk"]=newblk.cumsum()
    inv["block_in_bm"]=inv.groupby("BM_Programm")["blk"].transform(lambda s:pd.factorize(s)[0])
    inv["SOC"]=inv["block_in_bm"].map(lambda b:SOC_ORDER[b] if b<len(SOC_ORDER) else "ERR")
    inv["Capacity_Ah"]=inv["BM_Programm"].map(cap_map)
    inv["SOH"]=(inv["Capacity_Ah"]/soh_ref_ah*100).round(2)
    inv["chemistry"]=CHEMISTRY; inv["Battery_ID"]=BATTERY_ID
    inv["load_case"]=inv["SOC"].str.rstrip("%")+"_"+inv["Zustand"]+"_"+inv["C_rate"].astype(str)+"C"

    inv["load_case_in_scope"]=True
    if DROP_FIRST_PULSE:                                   # ① 每块首条 DCH-2C
        lo=int(round(NOMINAL_CURRENT_LEVELS_A.min()/ACN1_A))
        inv.loc[(inv.Zustand=="DCH")&(inv.C_rate==lo),"load_case_in_scope"]=False
    if APPLY_LOADCASE_REMOVAL:                             # ② Barai
        inv.loc[(inv.SOC=="90%")&(inv.Zustand=="CHA")&(inv.C_rate.isin(HIGH_CRATE_LEVELS)),"load_case_in_scope"]=False
        inv.loc[(inv.SOC=="10%")&(inv.Zustand=="DCH")&(inv.C_rate.isin(HIGH_CRATE_LEVELS)),"load_case_in_scope"]=False
    return pp, inv

pulse_points, pulse_inventory = build_pulse_inventory(RAW_FILE, cap_map, SOH_REF_AH)
print("总脉冲:",len(pulse_inventory),"| check-ups:",pulse_inventory["BM_Programm"].nunique(),
      "| in-scope:",int(pulse_inventory["load_case_in_scope"].sum()))

总脉冲: 414 | check-ups: 23 | in-scope: 253


In [4]:
# ============================ 前置静置（PAU，限定脉冲程序）→ 每脉冲最近前点 ============================
def collect_pause_for_pulses(path, inv):
    pau=read_filtered(
        path,
        targets=["PAU"],
        proc=PULSE_PROCEDURE,
        cols=["Time","Voltage","target","Prozedur","BM_Programm"]
    )
    pau["Time"]=pd.to_datetime(pau["Time"],utc=True)
    pau["Voltage"]=pd.to_numeric(pau["Voltage"],errors="coerce").astype(float)

    # 关联完整性保护：只允许 pulse 匹配同一 BM_Programm 内最近的前置 PAU。
    # merge_asof 要求时间键全局有序，因此按时间排序，并用 by 限制 BM。
    pau=(
        pau.rename(columns={"Time":"pause_time","Voltage":"pause_voltage"})
           [["BM_Programm","pause_time","pause_voltage"]]
           .sort_values("pause_time")
           .reset_index(drop=True)
    )
    st=(
        inv[["pulse_uid","BM_Programm","t0"]]
        .sort_values("t0")
        .reset_index(drop=True)
    )

    st=pd.merge_asof(
        st,
        pau,
        left_on="t0",
        right_on="pause_time",
        by="BM_Programm",
        direction="backward"
    )
    return st.set_index("pulse_uid")[["pause_time","pause_voltage"]].to_dict("index")

pause_map=collect_pause_for_pulses(RAW_FILE, pulse_inventory)

In [5]:
# ============================ R0/R1/R2 核心（移植 NCA calculate_r0_for_segment）============================
def _add_q(base,new): return new if base=="正常" else base+"；"+new
def _eff_start(gc):
    nz=np.flatnonzero(np.abs(gc)>ZERO_CURRENT_LIMIT); return int(nz[0]) if len(nz) else None

def calculate_r0_for_segment(group, esp, pause_time, pause_voltage):
    """按 NCA 冻结规格提取 R0/R1/R2。仅两处 LFP 适配：
       ① 前置 pause 由外部传入（PAU 行不在脉冲表里）；② 档位用吸附最近标称（免 float 误差）。"""
    group=group.sort_values("Time")
    o=dict(Nominal_Current_A=np.nan,R0=np.nan,R1=np.nan,R2=np.nan,
        CC_Valid_10s=pd.NA,CC_Valid_20s=pd.NA,CC_10s_Status=CC_NE,CC_20s_Status=CC_NE,
        CC_Max_Deviation_10s_pct=np.nan,CC_Max_Deviation_20s_pct=np.nan,
        R0_Target_Time=pd.NaT,Pause_to_Pulse_Time_Diff_s=np.nan,R0_Quality="正常")
    pst=group["Time"].iloc[0]; first_I=group["Current"].iloc[0]
    sabs=group["Current"].abs().dropna()
    if not sabs.empty:
        o["Nominal_Current_A"]=float(NOMINAL_CURRENT_LEVELS_A[np.argmin(np.abs(NOMINAL_CURRENT_LEVELS_A-sabs.max()))])
    if pd.isna(pause_time):
        o["R0_Quality"]="无法计算R0：无前置pause点"; return o
    gap=(pst-pause_time).total_seconds(); o["Pause_to_Pulse_Time_Diff_s"]=gap
    tgt=pause_time+pd.Timedelta(seconds=R0_TARGET_AFTER_PAUSE_SEC); o["R0_Target_Time"]=tgt
    if gap>MAX_PAUSE_TO_PULSE_GAP_SEC:
        o["R0_Quality"]=f"pause结束点到pulse首点时间差>{MAX_PAUSE_TO_PULSE_GAP_SEC:g}s"; return o
    if abs(first_I)<=ZERO_CURRENT_LIMIT:
        fv=group["Voltage"].iloc[0]
        if pd.notna(fv) and pd.notna(pause_voltage) and abs(fv-pause_voltage)>VOLTAGE_JUMP_LIMIT:
            o["R0_Quality"]="首点0且电压跳变"
    fps=R0_FIT_POINT_START
    if esp+1<len(group):
        v1=group["Voltage"].iloc[esp]; v2=group["Voltage"].iloc[esp+1]
        if pd.notna(v1) and pd.notna(v2) and np.isclose(float(v1),float(v2),rtol=0.0,atol=FIRST_TWO_VOLTAGE_EQUAL_ATOL):
            fps=3
    fs=esp+fps-1; fe=esp+R0_FIT_POINT_END
    fit=group.iloc[fs:fe].dropna(subset=["Time","Voltage","Current"])
    if len(fit)<2:
        o["R0_Quality"]=_add_q(o["R0_Quality"],"无法计算R0：拟合点不足"); return o
    fca=fit["Current"]
    lvl=float(NOMINAL_CURRENT_LEVELS_A[np.argmin(np.abs(NOMINAL_CURRENT_LEVELS_A-abs(fca.iloc[0])))])  # ← 吸附
    if fca.std()>STD_LIMIT.get(lvl,np.inf):
        o["R0_Quality"]=_add_q(o["R0_Quality"],"无法计算R0：早期拟合窗口电流不稳定"); return o
    eref = fca.median()
    effI = eref if (pd.notna(eref) and abs(eref) > ZERO_CURRENT_LIMIT) else group["Current"].iloc[esp]
    if abs(effI)<=ZERO_CURRENT_LIMIT:
        o["R0_Quality"]=_add_q(o["R0_Quality"],"无法计算R0：无非零有效电流"); return o
    if pd.isna(pause_voltage):
        o["R0_Quality"]=_add_q(o["R0_Quality"],"无法计算R0：pause电压缺失"); return o
    x=(fit["Time"]-tgt)/pd.Timedelta(seconds=1); y=fit["Voltage"].astype(float)
    slope,intercept=np.polyfit(x.to_numpy(),y.to_numpy(),1)
    o["R0"]=abs((intercept-pause_voltage)/effI)
    anc=group.iloc[esp:].dropna(subset=["Time","Voltage","Current"])
    rel=(anc["Time"]-group["Time"].iloc[esp])/pd.Timedelta(seconds=1)
    eref=fca.median(); fsr=(fit["Time"].min()-group["Time"].iloc[esp])/pd.Timedelta(seconds=1)
    def evaluate_cc_until(a):
        if len(anc)<2 or rel.max()<a: return CC_INSUF,np.nan
        if fsr>=a or pd.isna(eref) or abs(eref)<=ZERO_CURRENT_LIMIT: return CC_NE,np.nan
        wm=(rel>=fsr)&(rel<=a); wc=anc.loc[wm,"Current"].to_numpy()
        ac=np.interp(a,rel.to_numpy(),anc["Current"].to_numpy()); cv=np.append(wc,ac)
        if len(cv)<2: return CC_NE,np.nan
        md=np.max(np.abs(cv-eref))/abs(eref)*100
        return (CC_VALID,md) if md<=CC_CURRENT_DEVIATION_RATIO*100 else (CC_DEV,md)
    s10,m10=evaluate_cc_until(R1_ANCHOR_SEC); s20,m20=evaluate_cc_until(R2_ANCHOR_SEC)
    v10=s10==CC_VALID; v20=s20==CC_VALID
    o.update(CC_Valid_10s=v10,CC_Valid_20s=v20,CC_10s_Status=s10,CC_20s_Status=s20,
        CC_Max_Deviation_10s_pct=m10,CC_Max_Deviation_20s_pct=m20)
    if v10:
        vv=np.interp(R1_ANCHOR_SEC,rel.to_numpy(),anc["Voltage"].to_numpy())
        o["R1"]=abs((vv-pause_voltage)/effI)-o["R0"]
    if v20:
        a10,a20=np.interp([R1_ANCHOR_SEC,R2_ANCHOR_SEC],rel.to_numpy(),anc["Voltage"].to_numpy())
        o["R2"]=abs((a20-pause_voltage)/effI)-abs((a10-pause_voltage)/effI)
    return o

In [6]:
# ============================ 驱动：逐脉冲算 R0 → 长表（带 SOH/SOC）============================
rows=[]
for uid,grp in pulse_points.groupby("pulse_uid",sort=False):
    grp=grp.sort_values("Time"); esp=_eff_start(grp["Current"].to_numpy())
    if esp is None: 
        continue
    pm=pause_map.get(uid,{})
    r=calculate_r0_for_segment(grp,esp,pm.get("pause_time"),pm.get("pause_voltage"))
    r["pulse_uid"]=uid
    r["Current"]=float(grp["Current"].iloc[esp]); r["Voltage"]=float(grp["Voltage"].iloc[0])
    r["Time"]=grp["Time"].iloc[0]
    rows.append(r)
R=pd.DataFrame(rows)
R=R.merge(pulse_inventory[["pulse_uid","BM_Programm","ID","SOH","SOC","chemistry",
    "Battery_ID","Zustand","C_rate","load_case","load_case_in_scope"]],on="pulse_uid",how="left")
R["Zustand/Current"]=R["Zustand"]+"/"+R["Current"].round(1).astype(str)
for c in ["R0","R1","R2"]: R[c+"_mOhm"]=R[c]*1e3

R = R.sort_values(["BM_Programm","SOC","Zustand","C_rate"]).reset_index(drop=True)
R0_long = R[OUTPUT_COLUMNS].copy()
R0_long

,SOH,SOC,Battery_ID,Time,ID,Zustand,Zustand/Current,Current,Voltage,Nominal_Current_A,...,R1_mOhm,R2_mOhm,CC_Valid_10s,CC_Valid_20s,CC_Max_Deviation_10s_pct,CC_Max_Deviation_20s_pct,Pause_to_Pulse_Time_Diff_s,R0_Quality,load_case,load_case_in_scope
0,99.88,10%,A123_APR18650M1B_007,2024-12-19 18:51:48.190000+00:00,2_74,CHA,CHA/2.4,2.398792,3.251757,2.4,...,18.398225,4.361114,True,True,0.011257,0.015013,0.85,正常,10_CHA_2C,True
1,99.88,10%,A123_APR18650M1B_007,2024-12-19 20:54:35.400000+00:00,2_82,CHA,CHA/4.8,4.798417,3.299004,4.8,...,14.419956,3.291246,True,True,0.013130,0.016886,0.88,正常,10_CHA_4C,True
2,99.88,10%,A123_APR18650M1B_007,2024-12-19 22:58:43.130000+00:00,2_90,CHA,CHA/7.2,7.193287,3.335913,7.2,...,12.832531,3.110398,True,True,0.035011,0.035011,0.83,正常,10_CHA_6C,True
3,99.88,10%,A123_APR18650M1B_007,2024-12-19 17:50:44.990000+00:00,2_70,DCH,DCH/-2.4,-2.398788,3.138177,2.4,...,21.533397,6.160768,True,True,0.015013,0.015013,0.87,正常,10_DCH_2C,False
4,99.88,10%,A123_APR18650M1B_007,2024-12-19 19:52:51.710000+00:00,2_78,DCH,DCH/-4.8,-4.797858,3.125282,4.8,...,22.895388,9.118839,True,True,0.013135,0.013135,0.87,正常,10_DCH_4C,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
409,72.06,90%,A123_APR18650M1B_007,2026-05-08 05:32:33.700000+00:00,46_30,CHA,CHA/4.8,4.783474,3.439747,4.8,...,NaN,NaN,False,False,28.915479,52.714106,0.85,正常,90_CHA_4C,False
410,72.06,90%,A123_APR18650M1B_007,2026-05-08 07:36:21.190000+00:00,46_38,CHA,CHA/7.2,7.197247,3.339026,7.2,...,NaN,NaN,False,False,59.833966,NaN,0.83,正常,90_CHA_6C,False
411,72.06,90%,A123_APR18650M1B_007,2026-05-08 02:28:43.370000+00:00,46_18,DCH,DCH/-2.4,-2.397347,3.242449,2.4,...,14.897513,4.679138,True,True,0.041275,0.041275,0.85,正常,90_DCH_2C,False
412,72.06,90%,A123_APR18650M1B_007,2026-05-08 04:30:49.890000+00:00,46_26,DCH,DCH/-4.8,-4.798038,3.225997,4.8,...,18.854474,4.941884,True,True,0.011257,0.011257,0.88,正常,90_DCH_4C,True


In [7]:
# ============================ 自检 ============================
print("总脉冲:", len(R0_long), "| 输出列数:", R0_long.shape[1])
print("R0 有值 %d | R1 %d | R2 %d" % (
    R0_long["R0"].notna().sum(), R0_long["R1"].notna().sum(), R0_long["R2"].notna().sum()))
print("\nR0_Quality 分布:")
print(R0_long["R0_Quality"].value_counts().to_string())
print("\n=== R0/R1/R2 随 SOH 退化 (50%_DCH_4C) ===")
line = R0_long[R0_long["load_case"]=="50_DCH_4C"].sort_values("SOH", ascending=False)
print(line[["SOH","R0_mOhm","R1_mOhm","R2_mOhm"]].round(2).to_string(index=False))

总脉冲: 414 | 输出列数: 26
R0 有值 411 | R1 385 | R2 375

R0_Quality 分布:
R0_Quality
正常                        406
首点0且电压跳变                    5
pause结束点到pulse首点时间差>2s      3

=== R0/R1/R2 随 SOH 退化 (50%_DCH_4C) ===
  SOH  R0_mOhm  R1_mOhm  R2_mOhm
99.88    15.95    16.27     3.38
97.35    14.96    16.31     3.24
95.87    15.24    15.98     3.24
94.04    15.74    15.47     3.24
92.47    15.62    15.72     3.34
90.40    15.40    16.08     3.43
88.56    15.40    16.18     3.57
86.49    15.91    16.14     3.74
84.66    16.54    15.90     3.89
83.22    15.31    17.52     4.03
82.15    16.67    16.48     4.13
81.08    16.84    16.72     4.23
80.11    16.34    17.61     4.33
79.50    19.78    18.06     4.93
78.59    17.87    16.89     4.52
77.72    18.13    16.95     4.60
76.82    18.59    16.88     4.69
75.93    18.82    17.02     4.78
75.43    18.84    17.40     4.85
74.75    19.72    17.01     4.91
73.50    19.81    17.18     5.00
72.72    19.97    17.36     5.06
72.06    21.50    17.71     5.36


In [8]:
# ============================ 保存长表 ============================
OUT_CSV = Path("LFP_R0_长表.csv")
R0_long.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("saved:", OUT_CSV.resolve(), "| rows:", len(R0_long))

saved: D:\毕设数据\LFP\LFP_R0_长表.csv | rows: 414
